# RetailPulse -- Time Series Preparation for Demand Forecasting

**Objective:** Prepare daily sales data for time-series forecasting. Run stationarity tests (ADF, KPSS), perform seasonal decomposition, apply differencing transformations, and export Prophet/LSTM-ready datasets.


In [1]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)


In [2]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved: {name}")


## Load Daily Sales Data

In [3]:
daily = pd.read_csv(os.path.join(PROCESSED_DIR, "daily_sales_features.csv"),
                    parse_dates=["Date"])
daily = daily.sort_values("Date").reset_index(drop=True)

print(f"Shape: {daily.shape}")
print(f"Date range: {daily['Date'].min().date()} to {daily['Date'].max().date()}")
print(f"Total days: {len(daily)}")
print(f"Missing dates: {(daily['Date'].max() - daily['Date'].min()).days + 1 - len(daily)}")
daily.head()


Shape: (307, 22)
Date range: 2009-12-01 to 2010-12-09
Total days: 307
Missing dates: 67


,Date,total_revenue,total_quantity,transaction_count,unique_customers,avg_order_value,revenue_ma_7d,revenue_std_7d,qty_ma_7d,revenue_ma_14d,...,revenue_ma_30d,revenue_std_30d,qty_ma_30d,revenue_lag_1d,revenue_lag_7d,revenue_lag_14d,day_of_week,is_weekend,month,week_of_year
0,2009-12-01,43894.87,24335,98,91,20.284136,43894.870000,NaN,24335.000000,43894.870000,...,43894.870000,NaN,24335.000000,NaN,NaN,NaN,1,0,12,49
1,2009-12-02,52762.06,29679,110,94,23.491567,48328.465000,6270.050179,27007.000000,48328.465000,...,48328.465000,6270.050179,27007.000000,43894.87,NaN,NaN,2,0,12,49
2,2009-12-03,67413.62,48009,122,106,28.698859,54690.183333,11877.337458,34007.666667,54690.183333,...,54690.183333,11877.337458,34007.666667,52762.06,NaN,NaN,3,0,12,49
3,2009-12-04,33913.81,19954,80,76,17.943815,49496.090000,14211.328308,30494.250000,49496.090000,...,49496.090000,14211.328308,30494.250000,67413.62,NaN,NaN,4,0,12,49
4,2009-12-05,9803.05,5119,30,26,24.507625,41557.482000,21600.436896,25419.200000,41557.482000,...,41557.482000,21600.436896,25419.200000,33913.81,NaN,NaN,5,1,12,49


## Handle Missing Dates

Reindex to a continuous daily frequency. Missing days (e.g. no sales on weekends/holidays) are filled with zeros, since the absence of sales is meaningful for forecasting.


In [4]:
# Create continuous date range and reindex
full_range = pd.date_range(start=daily["Date"].min(), end=daily["Date"].max(), freq="D")
daily = daily.set_index("Date").reindex(full_range).rename_axis("Date").reset_index()

# Fill missing sales days with zeros for revenue and quantity
fill_zero_cols = ["total_revenue", "total_quantity", "transaction_count", "unique_customers"]
for col in fill_zero_cols:
    daily[col] = daily[col].fillna(0)

# Forward-fill the rolling/lag features, then fill remaining NaN
rolling_cols = [c for c in daily.columns if any(x in c for x in ["_ma_", "_std_", "_lag_"])]
for col in rolling_cols:
    daily[col] = daily[col].fillna(method="ffill").fillna(0)

# Recompute calendar features for newly added dates
daily["day_of_week"] = daily["Date"].dt.dayofweek
daily["is_weekend"] = daily["day_of_week"].isin([5, 6]).astype(int)
daily["month"] = daily["Date"].dt.month
daily["week_of_year"] = daily["Date"].dt.isocalendar().week.astype(int)

# Recompute avg_order_value
daily["avg_order_value"] = np.where(
    daily["transaction_count"] > 0,
    daily["total_revenue"] / daily["transaction_count"],
    0
)

print(f"After reindexing: {len(daily)} days (was {len(full_range)} expected)")
print(f"Zero-revenue days: {(daily['total_revenue'] == 0).sum()}")
daily.head()


After reindexing: 374 days (was 374 expected)
Zero-revenue days: 67


,Date,total_revenue,total_quantity,transaction_count,unique_customers,avg_order_value,revenue_ma_7d,revenue_std_7d,qty_ma_7d,revenue_ma_14d,...,revenue_ma_30d,revenue_std_30d,qty_ma_30d,revenue_lag_1d,revenue_lag_7d,revenue_lag_14d,day_of_week,is_weekend,month,week_of_year
0,2009-12-01,43894.87,24335.0,98.0,91.0,447.906837,43894.870000,0.000000,24335.000000,43894.870000,...,43894.870000,0.000000,24335.000000,0.00,0.0,0.0,1,0,12,49
1,2009-12-02,52762.06,29679.0,110.0,94.0,479.655091,48328.465000,6270.050179,27007.000000,48328.465000,...,48328.465000,6270.050179,27007.000000,43894.87,0.0,0.0,2,0,12,49
2,2009-12-03,67413.62,48009.0,122.0,106.0,552.570656,54690.183333,11877.337458,34007.666667,54690.183333,...,54690.183333,11877.337458,34007.666667,52762.06,0.0,0.0,3,0,12,49
3,2009-12-04,33913.81,19954.0,80.0,76.0,423.922625,49496.090000,14211.328308,30494.250000,49496.090000,...,49496.090000,14211.328308,30494.250000,67413.62,0.0,0.0,4,0,12,49
4,2009-12-05,9803.05,5119.0,30.0,26.0,326.768333,41557.482000,21600.436896,25419.200000,41557.482000,...,41557.482000,21600.436896,25419.200000,33913.81,0.0,0.0,5,1,12,49


## Time Series Visualization

In [5]:
fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)

# Revenue
axes[0].plot(daily["Date"], daily["total_revenue"], color="#bdc3c7", alpha=0.5, linewidth=0.8)
axes[0].plot(daily["Date"], daily["total_revenue"].rolling(7).mean(),
             color="#3498db", linewidth=1.5, label="7-day MA")
axes[0].plot(daily["Date"], daily["total_revenue"].rolling(30).mean(),
             color="#e74c3c", linewidth=2, label="30-day MA")
axes[0].set_ylabel("Revenue")
axes[0].set_title("Daily Revenue")
axes[0].legend()

# Transaction count
axes[1].plot(daily["Date"], daily["transaction_count"], color="#bdc3c7", alpha=0.5, linewidth=0.8)
axes[1].plot(daily["Date"], daily["transaction_count"].rolling(7).mean(),
             color="#2ecc71", linewidth=1.5, label="7-day MA")
axes[1].set_ylabel("Transactions")
axes[1].set_title("Daily Transaction Count")
axes[1].legend()

# Unique customers
axes[2].plot(daily["Date"], daily["unique_customers"], color="#bdc3c7", alpha=0.5, linewidth=0.8)
axes[2].plot(daily["Date"], daily["unique_customers"].rolling(7).mean(),
             color="#9b59b6", linewidth=1.5, label="7-day MA")
axes[2].set_ylabel("Customers")
axes[2].set_xlabel("Date")
axes[2].set_title("Daily Unique Customers")
axes[2].legend()

fig.suptitle("Daily Sales Metrics Overview", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "17_daily_sales_overview.png")
plt.show()


Saved: 17_daily_sales_overview.png


## Stationarity Tests

A time series is **stationary** if its statistical properties (mean, variance) do not change over time. Most forecasting models require or benefit from stationary data.

- **ADF (Augmented Dickey-Fuller):** Tests for unit root. H0 = non-stationary. Reject if p < 0.05.
- **KPSS (Kwiatkowski-Phillips-Schmidt-Shin):** Tests for stationarity. H0 = stationary. Reject if p < 0.05.


In [6]:
def run_stationarity_tests(series, name):
    """Run ADF and KPSS tests on a time series."""
    print(f"--- {name} ---")

    # ADF Test
    adf_result = adfuller(series.dropna(), autolag="AIC")
    adf_stat, adf_p = adf_result[0], adf_result[1]
    adf_conclusion = "Stationary" if adf_p < 0.05 else "Non-stationary"
    print(f"  ADF Statistic: {adf_stat:.4f}")
    print(f"  ADF p-value:   {adf_p:.6f}")
    print(f"  ADF Result:    {adf_conclusion}")

    # KPSS Test
    kpss_result = kpss(series.dropna(), regression="c", nlags="auto")
    kpss_stat, kpss_p = kpss_result[0], kpss_result[1]
    kpss_conclusion = "Stationary" if kpss_p > 0.05 else "Non-stationary"
    print(f"  KPSS Statistic: {kpss_stat:.4f}")
    print(f"  KPSS p-value:   {kpss_p:.4f}")
    print(f"  KPSS Result:    {kpss_conclusion}")

    # Combined interpretation
    if adf_conclusion == "Stationary" and kpss_conclusion == "Stationary":
        verdict = "Stationary (both tests agree)"
    elif adf_conclusion == "Non-stationary" and kpss_conclusion == "Non-stationary":
        verdict = "Non-stationary (both tests agree)"
    elif adf_conclusion == "Stationary" and kpss_conclusion == "Non-stationary":
        verdict = "Trend-stationary (difference-stationary)"
    else:
        verdict = "Inconclusive (tests disagree)"

    print(f"  Verdict:        {verdict}")
    print()

    return {
        "series": name,
        "adf_stat": round(adf_stat, 4),
        "adf_p": round(adf_p, 6),
        "kpss_stat": round(kpss_stat, 4),
        "kpss_p": round(kpss_p, 4),
        "verdict": verdict
    }


In [7]:
# Run tests on key series
revenue = daily["total_revenue"]
quantity = daily["total_quantity"]
transactions = daily["transaction_count"]

test_results = []
print("STATIONARITY TEST RESULTS")
print("=" * 60)
print()
test_results.append(run_stationarity_tests(revenue, "Total Revenue"))
test_results.append(run_stationarity_tests(quantity, "Total Quantity"))
test_results.append(run_stationarity_tests(transactions, "Transaction Count"))

results_df = pd.DataFrame(test_results)
results_df


STATIONARITY TEST RESULTS

--- Total Revenue ---
  ADF Statistic: -2.5650
  ADF p-value:   0.100454
  ADF Result:    Non-stationary
  KPSS Statistic: 1.7488
  KPSS p-value:   0.0100
  KPSS Result:    Non-stationary
  Verdict:        Non-stationary (both tests agree)

--- Total Quantity ---
  ADF Statistic: -3.4783
  ADF p-value:   0.008562
  ADF Result:    Stationary
  KPSS Statistic: 1.0120
  KPSS p-value:   0.0100
  KPSS Result:    Non-stationary
  Verdict:        Trend-stationary (difference-stationary)

--- Transaction Count ---
  ADF Statistic: -2.9051
  ADF p-value:   0.044759
  ADF Result:    Stationary
  KPSS Statistic: 1.7275
  KPSS p-value:   0.0100
  KPSS Result:    Non-stationary
  Verdict:        Trend-stationary (difference-stationary)



,series,adf_stat,adf_p,kpss_stat,kpss_p,verdict
0,Total Revenue,-2.5650,0.100454,1.7488,0.01,Non-stationary (both tests agree)
1,Total Quantity,-3.4783,0.008562,1.0120,0.01,Trend-stationary (difference-stationary)
2,Transaction Count,-2.9051,0.044759,1.7275,0.01,Trend-stationary (difference-stationary)


## Seasonal Decomposition

Decompose the revenue series into three components:
- **Trend:** Long-term direction of the series
- **Seasonal:** Repeating patterns at fixed intervals
- **Residual:** What's left after removing trend and seasonality


In [8]:
# Use only non-zero revenue days for decomposition
revenue_series = daily.set_index("Date")["total_revenue"]

# Additive decomposition with weekly period (7 days)
decomp_add = seasonal_decompose(revenue_series, model="additive", period=7)

fig, axes = plt.subplots(4, 1, figsize=(18, 14), sharex=True)

axes[0].plot(decomp_add.observed, color="#2c3e50", linewidth=0.8)
axes[0].set_ylabel("Observed")
axes[0].set_title("Original Series")

axes[1].plot(decomp_add.trend, color="#e74c3c", linewidth=1.5)
axes[1].set_ylabel("Trend")
axes[1].set_title("Trend Component")

axes[2].plot(decomp_add.seasonal, color="#3498db", linewidth=0.8)
axes[2].set_ylabel("Seasonal")
axes[2].set_title("Seasonal Component (period=7 days)")

axes[3].plot(decomp_add.resid, color="#2ecc71", linewidth=0.8, alpha=0.7)
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Date")
axes[3].set_title("Residual Component")

fig.suptitle("Seasonal Decomposition (Additive, Weekly Period)",
             fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "18_seasonal_decomposition_weekly.png")
plt.show()


Saved: 18_seasonal_decomposition_weekly.png


In [9]:
# Multiplicative decomposition with monthly period (30 days)
# Replace zeros with small value to avoid division issues in multiplicative mode
revenue_nonzero = revenue_series.replace(0, revenue_series[revenue_series > 0].min() * 0.1)
decomp_mult = seasonal_decompose(revenue_nonzero, model="multiplicative", period=30)

fig, axes = plt.subplots(4, 1, figsize=(18, 14), sharex=True)

axes[0].plot(decomp_mult.observed, color="#2c3e50", linewidth=0.8)
axes[0].set_ylabel("Observed")
axes[0].set_title("Original Series")

axes[1].plot(decomp_mult.trend, color="#e74c3c", linewidth=1.5)
axes[1].set_ylabel("Trend")
axes[1].set_title("Trend Component")

axes[2].plot(decomp_mult.seasonal, color="#3498db", linewidth=0.8)
axes[2].set_ylabel("Seasonal")
axes[2].set_title("Seasonal Component (period=30 days)")

axes[3].plot(decomp_mult.resid, color="#2ecc71", linewidth=0.8, alpha=0.7)
axes[3].axhline(1.0, color="#e74c3c", linestyle="--", alpha=0.5)
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Date")
axes[3].set_title("Residual Component")

fig.suptitle("Seasonal Decomposition (Multiplicative, Monthly Period)",
             fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "19_seasonal_decomposition_monthly.png")
plt.show()


Saved: 19_seasonal_decomposition_monthly.png


## Differencing for Stationarity

If the original series is non-stationary, apply first-order differencing (subtract previous day's value) to stabilize the mean.


In [10]:
# First-order differencing
daily["revenue_diff_1"] = daily["total_revenue"].diff(1)

# Seasonal differencing (7-day lag)
daily["revenue_diff_7"] = daily["total_revenue"].diff(7)

fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

axes[0].plot(daily["Date"], daily["total_revenue"], color="#3498db", alpha=0.7, linewidth=0.8)
axes[0].set_ylabel("Revenue")
axes[0].set_title("Original Revenue Series")

axes[1].plot(daily["Date"], daily["revenue_diff_1"], color="#e74c3c", alpha=0.7, linewidth=0.8)
axes[1].axhline(0, color="black", linestyle="--", alpha=0.3)
axes[1].set_ylabel("1st Difference")
axes[1].set_title("First-Order Differenced (d=1)")

axes[2].plot(daily["Date"], daily["revenue_diff_7"], color="#2ecc71", alpha=0.7, linewidth=0.8)
axes[2].axhline(0, color="black", linestyle="--", alpha=0.3)
axes[2].set_ylabel("7-Day Difference")
axes[2].set_xlabel("Date")
axes[2].set_title("Seasonal Differenced (d=7)")

fig.suptitle("Differencing Transformations", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "20_differencing.png")
plt.show()


Saved: 20_differencing.png


In [11]:
# Test stationarity of differenced series
print("STATIONARITY AFTER DIFFERENCING")
print("=" * 60)
print()
test_results.append(run_stationarity_tests(
    daily["revenue_diff_1"].dropna(), "Revenue (1st difference)"))
test_results.append(run_stationarity_tests(
    daily["revenue_diff_7"].dropna(), "Revenue (7-day difference)"))

results_df = pd.DataFrame(test_results)
results_df


STATIONARITY AFTER DIFFERENCING

--- Revenue (1st difference) ---
  ADF Statistic: -9.4496
  ADF p-value:   0.000000
  ADF Result:    Stationary
  KPSS Statistic: 0.3092
  KPSS p-value:   0.1000
  KPSS Result:    Stationary
  Verdict:        Stationary (both tests agree)

--- Revenue (7-day difference) ---
  ADF Statistic: -7.0151
  ADF p-value:   0.000000
  ADF Result:    Stationary
  KPSS Statistic: 0.0763
  KPSS p-value:   0.1000
  KPSS Result:    Stationary
  Verdict:        Stationary (both tests agree)



,series,adf_stat,adf_p,kpss_stat,kpss_p,verdict
0,Total Revenue,-2.5650,0.100454,1.7488,0.01,Non-stationary (both tests agree)
1,Total Quantity,-3.4783,0.008562,1.0120,0.01,Trend-stationary (difference-stationary)
2,Transaction Count,-2.9051,0.044759,1.7275,0.01,Trend-stationary (difference-stationary)
3,Revenue (1st difference),-9.4496,0.000000,0.3092,0.10,Stationary (both tests agree)
4,Revenue (7-day difference),-7.0151,0.000000,0.0763,0.10,Stationary (both tests agree)


## Autocorrelation Analysis (ACF / PACF)

ACF and PACF plots help identify the order of AR and MA components for ARIMA-type models, and reveal periodic patterns in the data.


In [12]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# ACF of original series
plot_acf(revenue.dropna(), lags=40, ax=axes[0, 0], alpha=0.05)
axes[0, 0].set_title("ACF -- Original Revenue")

# PACF of original series
plot_pacf(revenue.dropna(), lags=40, ax=axes[0, 1], alpha=0.05, method="ywm")
axes[0, 1].set_title("PACF -- Original Revenue")

# ACF of differenced series
diff_series = daily["revenue_diff_1"].dropna()
plot_acf(diff_series, lags=40, ax=axes[1, 0], alpha=0.05)
axes[1, 0].set_title("ACF -- Differenced Revenue (d=1)")

# PACF of differenced series
plot_pacf(diff_series, lags=40, ax=axes[1, 1], alpha=0.05, method="ywm")
axes[1, 1].set_title("PACF -- Differenced Revenue (d=1)")

fig.suptitle("Autocorrelation and Partial Autocorrelation Analysis",
             fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "21_acf_pacf.png")
plt.show()


Saved: 21_acf_pacf.png


## Prepare Data for Forecasting Models

### Prophet Format
Prophet requires a DataFrame with columns `ds` (date) and `y` (target value).

### LSTM Format
LSTM needs normalized sequences in sliding window format.


In [13]:
# Prophet-ready dataset
prophet_df = daily[["Date", "total_revenue"]].copy()
prophet_df.columns = ["ds", "y"]
prophet_df = prophet_df.sort_values("ds").reset_index(drop=True)

prophet_path = os.path.join(PROCESSED_DIR, "prophet_ready.csv")
prophet_df.to_csv(prophet_path, index=False)
print(f"Saved Prophet-ready data: {prophet_path}")
print(f"  Shape: {prophet_df.shape}")
print(f"  Date range: {prophet_df['ds'].min().date()} to {prophet_df['ds'].max().date()}")
prophet_df.head()


Saved Prophet-ready data: ../data/processed/prophet_ready.csv
  Shape: (374, 2)
  Date range: 2009-12-01 to 2010-12-09


,ds,y
0,2009-12-01,43894.87
1,2009-12-02,52762.06
2,2009-12-03,67413.62
3,2009-12-04,33913.81
4,2009-12-05,9803.05


In [14]:
# LSTM-ready dataset with features
lstm_cols = ["total_revenue", "total_quantity", "transaction_count",
             "unique_customers", "day_of_week", "is_weekend", "month"]

lstm_df = daily[["Date"] + lstm_cols].copy()
lstm_df = lstm_df.sort_values("Date").reset_index(drop=True)

lstm_path = os.path.join(PROCESSED_DIR, "lstm_ready.csv")
lstm_df.to_csv(lstm_path, index=False)
print(f"Saved LSTM-ready data: {lstm_path}")
print(f"  Shape: {lstm_df.shape}")
print(f"  Features: {lstm_cols}")
lstm_df.head()


Saved LSTM-ready data: ../data/processed/lstm_ready.csv
  Shape: (374, 8)
  Features: ['total_revenue', 'total_quantity', 'transaction_count', 'unique_customers', 'day_of_week', 'is_weekend', 'month']


,Date,total_revenue,total_quantity,transaction_count,unique_customers,day_of_week,is_weekend,month
0,2009-12-01,43894.87,24335.0,98.0,91.0,1,0,12
1,2009-12-02,52762.06,29679.0,110.0,94.0,2,0,12
2,2009-12-03,67413.62,48009.0,122.0,106.0,3,0,12
3,2009-12-04,33913.81,19954.0,80.0,76.0,4,0,12
4,2009-12-05,9803.05,5119.0,30.0,26.0,5,1,12


In [15]:
# Save updated daily features with differencing columns
updated_daily_path = os.path.join(PROCESSED_DIR, "daily_sales_features.csv")
daily.to_csv(updated_daily_path, index=False)
print(f"Updated daily_sales_features.csv with differencing columns")
print(f"  Shape: {daily.shape}")
print(f"  Total columns: {daily.shape[1]}")


Updated daily_sales_features.csv with differencing columns
  Shape: (374, 24)
  Total columns: 24


## Summary

In [16]:
print("TIME SERIES PREPARATION COMPLETE")
print("=" * 55)
print()
print(f"Daily series length: {len(daily)} days")
print(f"Date range: {daily['Date'].min().date()} to {daily['Date'].max().date()}")
print()
print("Stationarity Results:")
print("-" * 55)
for _, row in results_df.iterrows():
    print(f"  {row['series']:<30s} {row['verdict']}")
print()
print("Decomposition:")
print("  Additive (period=7):  weekly seasonality identified")
print("  Multiplicative (period=30): monthly seasonality identified")
print()
print("Exported files:")
print(f"  prophet_ready.csv          ({prophet_df.shape[0]} rows x {prophet_df.shape[1]} cols)")
print(f"  lstm_ready.csv             ({lstm_df.shape[0]} rows x {lstm_df.shape[1]} cols)")
print(f"  daily_sales_features.csv   ({daily.shape[0]} rows x {daily.shape[1]} cols)")
print()
print("Next steps:")
print("  - Baseline Prophet model for demand forecasting")
print("  - Train/test split strategy for time-series cross-validation")


TIME SERIES PREPARATION COMPLETE

Daily series length: 374 days
Date range: 2009-12-01 to 2010-12-09

Stationarity Results:
-------------------------------------------------------
  Total Revenue                  Non-stationary (both tests agree)
  Total Quantity                 Trend-stationary (difference-stationary)
  Transaction Count              Trend-stationary (difference-stationary)
  Revenue (1st difference)       Stationary (both tests agree)
  Revenue (7-day difference)     Stationary (both tests agree)

Decomposition:
  Additive (period=7):  weekly seasonality identified
  Multiplicative (period=30): monthly seasonality identified

Exported files:
  prophet_ready.csv          (374 rows x 2 cols)
  lstm_ready.csv             (374 rows x 8 cols)
  daily_sales_features.csv   (374 rows x 24 cols)

Next steps:
  - Baseline Prophet model for demand forecasting
  - Train/test split strategy for time-series cross-validation
